# Badge 4: Data Lake Workshop: Lesson 7: Exploring GeoSpatial Functions

Trail lengths, DENVER_AREA_TRAILS_2 and TRAILS_AND_BOUNDARIES.

**How to read this notebook**
- Each `## Page:` heading is one page of the Rise lesson, in course order.
- `> Course:` cells quote what the learner is told to do (text or screenshot steps).
- Code cells run that task. Course code is verbatim except role/warehouse names, swapped for `BADGE4_DLKW_ADMIN` (ACCOUNTADMIN), `BADGE4_DLKW_ROLE` (SYSADMIN) and `BADGE4_DLKW_WH` (COMPUTE_WH).
- **SQL equivalent** replaces a UI-only step; **From screenshot** marks code the course shows only as an image; **Derived solution** marks challenge answers the course does not give.
- DORA checks are not run here; see `dora/dora_tests.ipynb`.
- Files the course has learners download are on `@UTIL_DB.PUBLIC.BADGE4_DOWNLOADS`, standing in for the learner's Downloads folder.

Prerequisite: run `dora/account_setup.ipynb` once.

In [ ]:
# Isolation mapping (set once per notebook; SQL cells use these via {{...}})
ADMIN_ROLE    = "BADGE4_DLKW_ADMIN"   # stands in for the course's ACCOUNTADMIN
SYSADMIN_ROLE = "BADGE4_DLKW_ROLE"    # stands in for the course's SYSADMIN
WAREHOUSE     = "BADGE4_DLKW_WH"      # stands in for the course's COMPUTE_WH


def pdf(result):
    """A SQL cell's result as pandas (runtime 2.6+ returns a Snowpark DataFrame)."""
    return result.to_pandas() if hasattr(result, "to_pandas") else result

In [ ]:
%%sql -r session_context
use role {{SYSADMIN_ROLE}};
use secondary roles none;
use warehouse {{WAREHOUSE}};
use schema MELS_SMOOTHIE_CHALLENGE_DB.TRAILS;
select current_role(), current_warehouse(), current_database(), current_schema();

## Page: 🥋 Explore GeoSpatial Functions

> **Course:** 📓 Navigate to docs.snowflake.com and search for "GeoSpatial Functions". Outside Snowflake; nothing to run.

> ⚠️ **Audit note:** The course says this ST_LENGTH line won't work (DLKW_111). It runs as given and reports the error.

In [ ]:
%%sql -r st_length_without_conversion
execute immediate 'begin
  execute immediate ''--Remember this code? 
select 
''''LINESTRING(''''||
listagg(coord_pair, '''','''') 
within group (order by point_id)
||'''')'''' as my_linestring
,st_length(my_linestring) as length_of_trail --this line is new! but it won''''t work!
from cherry_creek_trail
group by trail_name'';
  return ''ran without error'';
exception when other then
  return ''Error (as the course predicts): '' || sqlerrm;
end;';

> **Course:** 🎯 TO_GEOGRAPHY Challenge Lab: add TO_GEOGRAPHY() so length_of_trail works (DLKW_112).

In [ ]:
%%sql -r st_length_with_to_geography
-- From screenshot DLKW_112, blur filled in
select 
'LINESTRING('||
listagg(coord_pair, ',') 
within group (order by point_id)
||')' as my_linestring
,st_length(to_geography(my_linestring)) as length_of_trail
from cherry_creek_trail
group by trail_name;

## Page: 🎯 Add Trail Length to Your View

In [ ]:
%%sql -r trail_lengths
-- From screenshot DLKW_113, blur filled in
select feature_name
, st_length(to_geography(whole_object))  as wo_length
, st_length(to_geography(geometry)) as geom_length
from denver_area_trails;

In [ ]:
%%sql -r get_ddl_view
select get_ddl('view', 'DENVER_AREA_TRAILS');

In [ ]:
%%sql -r replace_denver_area_trails
-- From screenshot DLKW_115 (column order), editing the view definition from get_ddl
create or replace view DENVER_AREA_TRAILS as
select
$1:features[0]:properties:Name::string as feature_name
,$1:features[0]:geometry:coordinates::string as feature_coordinates
,$1:features[0]:geometry::string as geometry
,st_length(to_geography(geometry)) as trail_length
,$1:features[0]:properties::string as feature_properties
,$1:crs:properties:name::string as specs
,$1 as whole_object
from @trails_geojson (file_format => ff_json);

In [ ]:
%%sql -r check_denver_area_trails
select * from DENVER_AREA_TRAILS;

## Page: 🥋 Make the Trail Data Align to Work Together

In [ ]:
%%sql -r create_denver_area_trails_2
--Create a view that will have similar columns to DENVER_AREA_TRAILS 
--Even though this data started out as Parquet, and we're joining it with geoJSON data
--So let's make it look like geoJSON instead.
create or replace view DENVER_AREA_TRAILS_2 as
select 
trail_name as feature_name
,'{"coordinates":['||listagg('['||lng||','||lat||']',',') within group (order by point_id)||'],"type":"LineString"}' as geometry
,st_length(to_geography(geometry))  as trail_length
from cherry_creek_trail
group by trail_name;

In [ ]:
%%sql -r union_all_trails
--Create a view that will have similar columns to DENVER_AREA_TRAILS 
select feature_name, geometry, trail_length
from DENVER_AREA_TRAILS
union all
select feature_name, geometry, trail_length
from DENVER_AREA_TRAILS_2;

## Page: 🥋 Enhancing the GeoSpatial Trail View

In [ ]:
%%sql -r geospatial_calcs
--Add more GeoSpatial Calculations to get more GeoSpecial Information! 
select feature_name
, to_geography(geometry) as my_linestring
, st_xmin(my_linestring) as min_eastwest
, st_xmax(my_linestring) as max_eastwest
, st_ymin(my_linestring) as min_northsouth
, st_ymax(my_linestring) as max_northsouth
, trail_length
from DENVER_AREA_TRAILS
union all
select feature_name
, to_geography(geometry) as my_linestring
, st_xmin(my_linestring) as min_eastwest
, st_xmax(my_linestring) as max_eastwest
, st_ymin(my_linestring) as min_northsouth
, st_ymax(my_linestring) as max_northsouth
, trail_length
from DENVER_AREA_TRAILS_2;

> **Course:** 🥋 Make it a View (DLKW_122): add `create view trails_and_boundaries as` above the select.

In [ ]:
%%sql -r create_trails_and_boundaries
-- From screenshot DLKW_122
create view trails_and_boundaries as
--Add more GeoSpatial Calculations to get more GeoSpecial Information! 
select feature_name
, to_geography(geometry) as my_linestring
, st_xmin(my_linestring) as min_eastwest
, st_xmax(my_linestring) as max_eastwest
, st_ymin(my_linestring) as min_northsouth
, st_ymax(my_linestring) as max_northsouth
, trail_length
from DENVER_AREA_TRAILS
union all
select feature_name
, to_geography(geometry) as my_linestring
, st_xmin(my_linestring) as min_eastwest
, st_xmax(my_linestring) as max_eastwest
, st_ymin(my_linestring) as min_northsouth
, st_ymax(my_linestring) as max_northsouth
, trail_length
from DENVER_AREA_TRAILS_2;

In [ ]:
%%sql -r select_trails_and_boundaries
select * from trails_and_boundaries;

## Page: 🥋 Creating A Bounding Box Polygon

In [ ]:
%%sql -r bounding_box
select 'POLYGON(('|| 
    min(min_eastwest)||' '||max(max_northsouth)||','|| 
    max(max_eastwest)||' '||max(max_northsouth)||','|| 
    max(max_eastwest)||' '||min(min_northsouth)||','|| 
    min(min_eastwest)||' '||min(min_northsouth)||'))' AS my_polygon
from trails_and_boundaries;

## Page: 🤖 DORA DLKW07

### 🤖 DORA check DLKW07 goes here
The course has the learner run **DLKW07** (trails northern extent) at this point.
It is in `dora/dora_tests.ipynb`, section **DLKW07**, as a local check plus the exact DORA code.